## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
from datetime import datetime
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from funciones import *

In [2]:
# Abrir archivo clean_data
data_folder = "../data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   Beta                            502 non-null    float64 
 1   DividendYield                   502 non-null    float64 
 2   ReturnOnAssets                  502 non-null    float64 
 3   profitMargins                   502 non-null    float64 
 4   operatingMargins                502 non-null    float64 
 5   currentRatio                    502 non-null    float64 
 6   revenueGrowth                   502 non-null    float64 
 7   shortPercentOfFloat             502 non-null    float64 
 8   PriceToBook_Transformed         502 non-null    float64 
 9   returnOnEquity_Transformed      502 non-null    float64 
 10  ForwardPE_Transformed           502 non-null    float64 
 11  MarketCap_log                   502 non-null    float64 
 12  debtToEquity_log                5

# Modelo inicial: Regresión Lineal

Se genera en primer lugar un modelo de regresion lineal a fin de comprobar la relevancia de las variables.

Se analizan primero las relaciones de variables continuas con la variable objetivo mediante la V de Cramer:

In [3]:
# Se generan 2 variables aleatorias para referencia
df['random1'] = np.random.uniform(0,1,size=df.shape[0])
df['random2'] = np.random.uniform(0,1,size=df.shape[0])

# Se separan las variables objetivo y los inputs de predictores
varObjCont = df['MarketCap_log']

imput = df.drop(['MarketCap_log', 'Ticker'],axis=1)

# Ranking V de Cramer vs. obj continua
tablaCramer = pd.DataFrame(imput.apply(lambda x: cramers_v(x,varObjCont)),columns=['VCramer'])

px.bar(tablaCramer,x=tablaCramer.VCramer,title='Relaciones frente al MarketCap').update_yaxes(categoryorder="total ascending")

* Se observa una gran relevancia de la variable `shortPercentOfFloat`.
* Las variables `debtToEquity_log` y `trailingPegRatio_log` muestran una relacion baja, inferior a las variables generadas de forma aleatoria.

In [4]:
# Se eliminan random1 y random2 luego de contrastar su baja significancia en el modelo
df = df.drop(['random1', 'random2'], axis=1)

In [5]:
# Formula Primer Modelo OLS: Modelo Completo
form = ols_formula(df, "MarketCap_log", 'Ticker')
form

'MarketCap_log ~ Beta + DividendYield + ReturnOnAssets + profitMargins + operatingMargins + currentRatio + revenueGrowth + shortPercentOfFloat + PriceToBook_Transformed + returnOnEquity_Transformed + ForwardPE_Transformed + debtToEquity_log + trailingPegRatio_log + EnterpriseToEbitda_Transformed + Sector'

In [6]:
modelo1 = ols(form,data=df).fit()
print(modelo1.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.502
Model:                            OLS   Adj. R-squared:                  0.477
Method:                 Least Squares   F-statistic:                     20.05
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           1.40e-57
Time:                        10:58:52   Log-Likelihood:                -595.16
No. Observations:                 502   AIC:                             1240.
Df Residuals:                     477   BIC:                             1346.
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

A continuacion se reducen las variables no significativas segun sus p valores.

In [7]:
# Segundo Modelo: reduciendo debtToEquity_log y trailingPegRatio_log
form2 = ols_formula(df, "MarketCap_log", 'Ticker', 'debtToEquity_log', 'trailingPegRatio_log')
modelo2 = ols(form2,data=df).fit()
print(modelo2.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.502
Model:                            OLS   Adj. R-squared:                  0.479
Method:                 Least Squares   F-statistic:                     21.96
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           6.67e-59
Time:                        10:58:52   Log-Likelihood:                -595.21
No. Observations:                 502   AIC:                             1236.
Df Residuals:                     479   BIC:                             1333.
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

* Se mantuvo el ajuste del modelo al eliminar las variables.

In [8]:
# Tercer Modelo: reduciendo currentRatio y ReturnOnAssets
form3 = ols_formula(df, "MarketCap_log", 'Ticker', 'debtToEquity_log', 'trailingPegRatio_log', 'currentRatio', 'ReturnOnAssets')
modelo3 = ols(form3,data=df).fit()
print(modelo3.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.502
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     24.20
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           3.50e-60
Time:                        10:58:52   Log-Likelihood:                -595.48
No. Observations:                 502   AIC:                             1233.
Df Residuals:                     481   BIC:                             1322.
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

* Se mantuvo la bondad de ajuste del modelo.

In [9]:
# Cuarto Modelo: reduciendo operatingMargins
form4 = ols_formula(df, "MarketCap_log", 'Ticker', 'debtToEquity_log', 'trailingPegRatio_log', 'currentRatio', 'ReturnOnAssets', 'operatingMargins')
modelo4 = ols(form4,data=df).fit()
print(modelo4.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.501
Model:                            OLS   Adj. R-squared:                  0.482
Method:                 Least Squares   F-statistic:                     25.50
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           7.78e-61
Time:                        10:58:52   Log-Likelihood:                -595.62
No. Observations:                 502   AIC:                             1231.
Df Residuals:                     482   BIC:                             1316.
Df Model:                          19                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

* Si bien se redujo el R2, mejoro el R2 Ajustado al quitar la variable.

In [16]:
# Quinto Modelo: reduciendo revenueGrowth
form5 = ols_formula(df, "MarketCap_log", 'Ticker', 'debtToEquity_log', 'trailingPegRatio_log', 'currentRatio', 'ReturnOnAssets', 'operatingMargins', 'revenueGrowth')
modelo5 = ols(form5,data=df).fit()
print(modelo5.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.498
Model:                            OLS   Adj. R-squared:                  0.479
Method:                 Least Squares   F-statistic:                     26.62
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           6.78e-61
Time:                        11:32:43   Log-Likelihood:                -597.26
No. Observations:                 502   AIC:                             1233.
Df Residuals:                     483   BIC:                             1313.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

* Se redujo el R2 ajustado al eliminar la variable. 

Se evalua entonces el modelo 4 como optima relacion entre bondad de ajuste y parsimonia.
Las variables que resultaron relevantes son:
* `shortPercentOfFloat`: domina la regresion lineal con un estadistico *t* de -14.205 (la distancia a cero es de 14 desviaciones estandar)
* `sector`: las categorias `T.Communication Services`, `T.Energy` y  `T.Real Estate` obtienen p valores significativos.
* `profitMargins`.
* `ForwardPE_Transformed`.
* `EnterpriseToEbitda_Transformed`.
* `DividendYield`.
* `PriceToBook_Transformed`.

# Modelo de ensamblado de arboles RandomForest

In [11]:
# definir matrices de entrada y objetivo
X = df.drop(['MarketCap_log','Ticker'], axis=1)
y = df['MarketCap_log']

# identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        max_samples= None        
        ))
])

pipe.fit(X_train, y_train)

# comparar predicciones con el conjunto test


y_pred = pipe.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.618353021928469
R2: 0.6043756106564728


In [12]:
# Reajustar modelo con todos los datos
print("Reajustando modelo con el dataset completo...")

X_completo = df.drop(['MarketCap_log','Ticker'], axis=1)
y_completo = df['MarketCap_log']

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        max_samples= None        
        ))
])

pipe_final.fit(X_completo, y_completo)
r2_final = pipe_final.score(X_completo, y_completo)
print(f"R² sobre dataset completo: {r2_final:.4f}")

Reajustando modelo con el dataset completo...
R² sobre dataset completo: 0.9460


In [15]:
# Importancia de factores en el modelo
rf_model = pipe_final.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe_final.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
7,num__shortPercentOfFloat,0.294412
8,num__PriceToBook_Transformed,0.089858
13,num__EnterpriseToEbitda_Transformed,0.084684
3,num__profitMargins,0.065861
2,num__ReturnOnAssets,0.057036
10,num__ForwardPE_Transformed,0.054359
6,num__revenueGrowth,0.052711
4,num__operatingMargins,0.050000
9,num__returnOnEquity_Transformed,0.049078
1,num__DividendYield,0.045164


* Se observa tambien la gran relevancia de la variable `shortPercentOfFloat`.

In [17]:
# Test de validación cruzada
cross_val_scores = cross_val_score(pipe_final, X_completo, y_completo, cv=5, scoring='r2')
print(f"R² promedio CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio CV: 0.5369 ± 0.0824


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Overprediction: si los residuos son mayores o iguales a cero.
*  Underprediction: si los residuos son menores a cero.

In [30]:
# Cluster: sobre/sub prediccion respecto a la identidad
residuos = y_pred - y_test
cluster = pd.Series(['Overprediction' if r >= 0 else 'Underprediction' for r in residuos], 
                     index=y_test.index)

# Visualizar con los 2 clusters
fig = px.scatter(x=y_test, y=y_pred, color=cluster,
                 labels={'x':'Real','y':'Predicho','color':'Cluster'},
                 title='Predicciones vs Reales (Clusters)')
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
             x1=y_test.max(), y1=y_test.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas por cluster:")
print(f"Overprediction: {(cluster == 'Overprediction').sum()} casos, residuo medio: {residuos[cluster == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(cluster == 'Underprediction').sum()} casos, residuo medio: {residuos[cluster == 'Underprediction'].mean():.4f}")


Estadísticas por cluster:
Overprediction: 62 casos, residuo medio: 0.2823
Underprediction: 64 casos, residuo medio: -0.4154


In [31]:
# Recrear clusters con predicciones del modelo final
y_pred_completo = pipe_final.predict(X_completo)
residuos_completo = y_pred_completo - y_completo
cluster_completo = pd.Series(['Overprediction' if r > 0 else 'Underprediction' for r in residuos_completo], 
                              index=y_completo.index)

# Actualizar columna de cluster y calcular distancia a la identidad
df['cluster_prediction'] = cluster_completo.values
# Distancia perpendicular a la recta identidad (y = x)
df['distance_identity'] = residuos_completo / np.sqrt(2)

# Visualizar con plotly
fig = px.scatter(df, x='MarketCap_log', y=y_pred_completo, color='cluster_prediction',
                 hover_data=['Ticker','Sector'],
                 labels={'x':'MarketCap (Real)','y':'MarketCap (Predicho)','cluster_prediction':'Cluster'},
                 title='Clusters Overprediction/Underprediction (Datos Completos)')
fig.add_shape(type='line', x0=y_completo.min(), y0=y_completo.min(),
             x1=y_completo.max(), y1=y_completo.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas de clusters (datos completos):")
print(f"Overprediction: {(df['cluster_prediction'] == 'Overprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(df['cluster_prediction'] == 'Underprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Underprediction'].mean():.4f}")


Estadísticas de clusters (datos completos):
Overprediction: 270 casos, residuo medio: 0.0947
Underprediction: 232 casos, residuo medio: -0.1121


In [32]:
print("\nDataframe con cluster:")
print(df[['Ticker', 'MarketCap_log', 'distance_identity','cluster_prediction']].sort_values('distance_identity', ascending=False).head(10))


Dataframe con cluster:
    Ticker  MarketCap_log  distance_identity cluster_prediction
194    FRT       2.346902           0.322805     Overprediction
395    RMD       3.539583           0.222737     Overprediction
467   VLTO       3.154221           0.220936     Overprediction
221     GL       2.528923           0.215046     Overprediction
283     LW       1.921147           0.205977     Overprediction
450    TRV       4.186949           0.205154     Overprediction
337   NDSN       2.796901           0.197689     Overprediction
372   POOL       2.194469           0.179306     Overprediction
250   PODD       2.702896           0.174768     Overprediction
17     LNT       2.978216           0.173381     Overprediction


In [33]:
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

df.to_csv(f"{data_folder}/cluster_data_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimizacion de hiperparametros

* La siguiente celda se utilizó para optimizar los hiperparámetros de los modelos de forma individual. La ejecución de la misma no es necesaria.
* En caso de utilizarla, reemplazar 'no' por 'si' en la primer línea. Puede demorar varios minutos.

In [ ]:
ejecutar_celda = 'no' # Reemplazar para ejecutar
modelo_a_optimizar = 1  # 1 = Random Forest 

if ejecutar_celda == 'si':

    if modelo_a_optimizar == 1:
        nombre_modelo= 'Random Forest'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para Random Forest
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocessor),
            ('rf', RandomForestRegressor(random_state=42))
        ])

        param_grid = {
            'rf__n_estimators': [300],
            'rf__max_depth': [25],
            'rf__min_samples_leaf': [1],
            'rf__max_features': ['sqrt'],
            'rf__max_samples': [None]
        }

    elif modelo_a_optimizar == 2: # Por implementar
        nombre_modelo= 'LightGBM'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para LightGBM
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('lgb', lgb.LGBMRegressor(random_state=42,
                                      subsample=0.8,
                                      colsample_bytree=0.8
))
        ])

        param_grid = {
            'lgb__n_estimators': [500, 550, 600],
            'lgb__max_depth': [5],
            'lgb__learning_rate': [0.05],
            'lgb__num_leaves': [20],
            'lgb__min_child_samples': [20]
        }

    else:
        raise ValueError("Opción no válida. Por favor, selecciona 1 o 2.")

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='neg_root_mean_squared_error',
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con X_train y y_train originales
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X_train, y_train)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)

    # Extraer el mejor modelo
    best_model = grid_search.best_estimator_

    # Predecir en el set de prueba usando X_test original
    y_pred = best_model.predict(X_test)

    print("\nEvaluación en el set de Prueba:\n", obtener_metricas(y_test, y_pred, nombre_modelo))

Configurando GridSearchCV para Random Forest
Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.
Fitting 3 folds for each of 3 candidates, totalling 9 fits

--- Búsqueda Finalizada ---
Mejores hiperparámetros encontrados:
{'rf__max_depth': 25, 'rf__max_features': 'sqrt', 'rf__max_samples': None, 'rf__min_samples_leaf': 1, 'rf__n_estimators': 300}

Evaluación en el set de Prueba:
 {'Modelo': 'Random Forest', 'MAE': 0.6138621904099874, 'MSE': 0.618353021928469, 'RMSE': np.float64(0.7863542598145374), 'R2': 0.6043756106564728}
